# Classificação de Hipotireoidismo com Machine Learning

## Tech Challenge - Fase 1 | IA para Devs

Este notebook apresenta uma solução inicial de Machine Learning aplicada à saúde, utilizando dados tabulares para apoiar a análise inicial de exames relacionados à tireoide. O objetivo é construir e comparar modelos de classificação capazes de estimar se um registro clínico indica **hipotireoidismo** ou resultado **negativo**.

O problema tratado é uma classificação binária: a partir de variáveis clínicas, demográficas e laboratoriais, o modelo busca prever a classe do paciente. Em um contexto hospitalar, esse tipo de solução pode apoiar processos de triagem, priorização de análise e organização de fluxos clínicos.

Este projeto tem finalidade acadêmica e exploratória. O modelo deve ser entendido como ferramenta de apoio à decisão médica, e não como diagnóstico automatizado definitivo. A decisão clínica final sempre deve ser tomada por profissionais da saúde.

## 1. Preparação do Ambiente

O notebook foi preparado para execução local e no Google Colab. Caso alguma biblioteca não esteja disponível no Colab, execute a célula opcional abaixo.

In [ ]:
# Célula opcional para Google Colab.
# Execute somente se alguma biblioteca não estiver disponível no ambiente.
#
# !pip install -q pandas numpy matplotlib seaborn scikit-learn joblib shap

## 2. Importação das Bibliotecas

Importamos bibliotecas para manipulação dos dados, visualização, modelagem, avaliação e salvamento do modelo. O uso de `RANDOM_STATE` melhora a reprodutibilidade dos experimentos.

In [ ]:
import warnings
from pathlib import Path
from IPython.display import display

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, auc, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import learning_curve, train_test_split, validation_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

RANDOM_STATE = 42
TARGET_COLUMN = "target"
POSITIVE_LABEL = "hypothyroid"
NEGATIVE_LABEL = "negative"

## 3. Caminhos do Projeto

A estrutura esperada do repositório permite execução local e no Colab. Se o repositório for clonado no Colab, o caminho relativo continuará funcionando quando o notebook estiver na pasta `notebooks` ou quando a execução ocorrer a partir da raiz.

In [ ]:
DATA_PATH = Path("../dataset/hypothyroid_final.csv")
# Alternativa comum no Google Colab:
# DATA_PATH = Path("/content/TechCahllenge_Tireoide/dataset/hypothyroid_final.csv")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

candidate_paths = [
    DATA_PATH,
    PROJECT_ROOT / "dataset" / "hypothyroid_final.csv",
    Path("/content/TechCahllenge_Tireoide/dataset/hypothyroid_final.csv"),
]

resolved_data_path = next((path for path in candidate_paths if path.exists()), None)
if resolved_data_path is None:
    raise FileNotFoundError("Dataset não encontrado. Ajuste DATA_PATH para hypothyroid_final.csv.")

DATA_PATH = resolved_data_path
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATA_PATH.resolve()}")
print(f"Modelos: {MODELS_DIR.resolve()}")
print(f"Figuras: {FIGURES_DIR.resolve()}")

## 4. Carregamento do Dataset

O dataset já foi preparado a partir dos arquivos originais `.data` e `.names`. A coluna `target` representa a classe a ser prevista.

In [ ]:
dados = pd.read_csv(DATA_PATH, na_values=["?", "", "NA", "NaN"])

print("Primeiras linhas:")
display(dados.head())
print(f"Dimensão: {dados.shape[0]} linhas e {dados.shape[1]} colunas")
print("\nColunas:")
print(list(dados.columns))

In [ ]:
print("Informações gerais:")
dados.info()

print("\nEstatísticas numéricas:")
display(dados.describe().T)

print("\nEstatísticas categóricas:")
display(dados.describe(include="object").T)

**Observação inicial:** a base contém variáveis contínuas, como idade e exames laboratoriais, além de variáveis categóricas/binárias relacionadas a histórico clínico, medicamentos, sintomas e indicação de exames medidos.

## 5. Análise Exploratória dos Dados (EDA)

A EDA ajuda a entender estrutura, qualidade, distribuição e possíveis relações entre variáveis. Em dados médicos, valores ausentes, desbalanceamento de classes e valores extremos podem carregar informação clínica relevante.

In [ ]:
linhas, colunas = dados.shape
print(f"Linhas: {linhas}")
print(f"Colunas: {colunas}")

resumo_tipos = dados.dtypes.value_counts().rename_axis("tipo").reset_index(name="quantidade")
display(resumo_tipos)

plt.figure(figsize=(7, 4))
sns.barplot(data=resumo_tipos, x="tipo", y="quantidade")
plt.title("Quantidade de variáveis por tipo de dado")
plt.xlabel("Tipo de dado")
plt.ylabel("Quantidade de colunas")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

O gráfico de tipos de dados mostra se as variáveis foram interpretadas corretamente. Variáveis numéricas e categóricas receberão tratamentos diferentes no pipeline.

In [ ]:
missing_count = dados.isna().sum().sort_values(ascending=False)
missing_percent = (missing_count / len(dados) * 100).round(2)
missing_table = pd.DataFrame({"ausentes": missing_count, "percentual": missing_percent})
missing_table = missing_table[missing_table["ausentes"] > 0]
display(missing_table)

plt.figure(figsize=(12, 5))
sns.heatmap(dados.isna(), cbar=False, cmap="viridis", yticklabels=False)
plt.title("Mapa de valores ausentes no dataset")
plt.xlabel("Variáveis")
plt.ylabel("Registros")
plt.tight_layout()
plt.show()

if not missing_table.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=missing_table.reset_index(), x="percentual", y="index", color="#5DADE2")
    plt.title("Percentual de valores ausentes por variável")
    plt.xlabel("Percentual de valores ausentes (%)")
    plt.ylabel("Variável")
    plt.tight_layout()
    plt.show()

Valores ausentes são comuns em bases médicas, pois nem todos os exames são solicitados para todos os pacientes. Por isso, a imputação será feita dentro do pipeline, aprendendo parâmetros apenas com o treino.

In [ ]:
duplicados = dados.duplicated().sum()
print(f"Registros duplicados encontrados: {duplicados}")
print(f"Percentual de duplicados: {duplicados / len(dados) * 100:.2f}%")

Registros duplicados podem distorcer treinamento e avaliação, especialmente se o mesmo registro aparecer em conjuntos diferentes. Eles serão tratados na limpeza.

In [ ]:
target_counts = dados[TARGET_COLUMN].value_counts()
target_percent = dados[TARGET_COLUMN].value_counts(normalize=True).mul(100).round(2)
target_summary = pd.DataFrame({"quantidade": target_counts, "percentual": target_percent})
display(target_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=dados, x=TARGET_COLUMN, order=target_counts.index, ax=axes[0])
axes[0].set_title("Distribuição da variável-alvo")
axes[0].set_xlabel("Classe")
axes[0].set_ylabel("Quantidade")
axes[1].pie(target_counts.values, labels=target_counts.index, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Proporção das classes")
plt.tight_layout()
plt.show()

A distribuição mostra desbalanceamento importante entre classes. Em saúde, isso exige cuidado porque a acurácia pode parecer alta mesmo quando o modelo falha em identificar a classe clinicamente mais sensível.

In [ ]:
numeric_columns = dados.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = [col for col in dados.columns if col not in numeric_columns and col != TARGET_COLUMN]

print("Variáveis numéricas:", numeric_columns)
print("\nVariáveis categóricas:", categorical_columns)

In [ ]:
if numeric_columns:
    n_cols = 3
    n_rows = int(np.ceil(len(numeric_columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, numeric_columns):
        sns.histplot(data=dados, x=col, kde=True, ax=ax, color="#4C78A8")
        ax.set_title(f"Distribuição de {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Frequência")
    for ax in axes[len(numeric_columns):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

Histogramas e curvas KDE permitem observar assimetria, concentração de valores e possíveis extremos. Em exames laboratoriais, distribuições assimétricas são esperadas.

In [ ]:
if numeric_columns:
    n_cols = 3
    n_rows = int(np.ceil(len(numeric_columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, numeric_columns):
        sns.boxplot(data=dados, x=TARGET_COLUMN, y=col, ax=ax, palette="Set3")
        ax.set_title(f"Boxplot de {col} por classe")
        ax.set_xlabel("Classe")
        ax.set_ylabel(col)
        ax.tick_params(axis="x", rotation=20)
    for ax in axes[len(numeric_columns):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
selected_numeric = [col for col in ["age", "TSH", "T3", "TT4", "T4U", "FTI"] if col in dados.columns]

if selected_numeric:
    n_cols = 2
    n_rows = int(np.ceil(len(selected_numeric) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4.5 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, selected_numeric):
        sns.violinplot(data=dados, x=TARGET_COLUMN, y=col, inner="quartile", ax=ax, palette="Pastel1")
        ax.set_title(f"Violin plot de {col} por classe")
        ax.set_xlabel("Classe")
        ax.set_ylabel(col)
        ax.tick_params(axis="x", rotation=20)
    for ax in axes[len(selected_numeric):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

Boxplots e violin plots mostram dispersão, mediana, quartis e possíveis outliers por classe. Em problemas médicos, extremos não devem ser descartados automaticamente.

In [ ]:
selected_categorical = [
    col for col in [
        "sex", "on_thyroxine", "query_hypothyroid", "query_hyperthyroid",
        "pregnant", "sick", "tumor", "thyroid_surgery", "TSH_measured", "T3_measured"
    ] if col in dados.columns
]

if selected_categorical:
    n_cols = 2
    n_rows = int(np.ceil(len(selected_categorical) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4.2 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, selected_categorical):
        sns.countplot(data=dados, x=col, hue=TARGET_COLUMN, ax=ax)
        ax.set_title(f"Distribuição de {col} por classe")
        ax.set_xlabel(col)
        ax.set_ylabel("Quantidade")
        ax.tick_params(axis="x", rotation=20)
        ax.legend(title="Classe")
    for ax in axes[len(selected_categorical):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
if selected_categorical:
    percent_frames = []
    for col in selected_categorical:
        temp = (
            pd.crosstab(dados[col], dados[TARGET_COLUMN], normalize="index")
            .mul(100)
            .reset_index()
            .melt(id_vars=col, var_name="classe", value_name="percentual")
            .rename(columns={col: "categoria"})
        )
        temp["variavel"] = col
        percent_frames.append(temp)

    categorical_percent = pd.concat(percent_frames, ignore_index=True)
    display(categorical_percent.head(20))

    plt.figure(figsize=(13, 7))
    plot_data = categorical_percent[categorical_percent["classe"] == POSITIVE_LABEL]
    sns.barplot(data=plot_data, x="percentual", y="variavel", hue="categoria")
    plt.title("Percentual da classe positiva por categoria")
    plt.xlabel("Percentual de hipotireoidismo dentro da categoria (%)")
    plt.ylabel("Variável")
    plt.legend(title="Categoria", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

Os gráficos categóricos mostram se certas condições, histórico clínico ou indicadores de exames medidos aparecem com maior frequência entre registros positivos. Associação estatística não significa causalidade.

In [ ]:
scatter_features = [col for col in ["TSH", "T3", "TT4", "T4U", "FTI", "age"] if col in dados.columns]

if len(scatter_features) >= 2:
    sns.pairplot(
        dados[[TARGET_COLUMN] + scatter_features].dropna(),
        hue=TARGET_COLUMN,
        diag_kind="kde",
        corner=True,
        plot_kws={"alpha": 0.55, "s": 22},
    )
    plt.suptitle("Relações entre variáveis laboratoriais selecionadas", y=1.02)
    plt.show()

In [ ]:
correlation_data = dados.copy()
correlation_data["target_binario"] = correlation_data[TARGET_COLUMN].map({NEGATIVE_LABEL: 0, POSITIVE_LABEL: 1})
corr_columns = numeric_columns + ["target_binario"]
corr_matrix = correlation_data[corr_columns].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Heatmap de correlação entre variáveis numéricas")
plt.tight_layout()
plt.show()

corr_target = (
    corr_matrix["target_binario"]
    .drop("target_binario")
    .dropna()
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
display(corr_target.to_frame("correlacao_com_alvo"))

plt.figure(figsize=(9, 5))
sns.barplot(x=corr_target.values, y=corr_target.index, palette="vlag")
plt.axvline(0, color="black", linewidth=1)
plt.title("Ranking de correlação com a classe positiva")
plt.xlabel("Correlação com target binário")
plt.ylabel("Variável numérica")
plt.tight_layout()
plt.show()

A correlação mede associação linear entre variáveis numéricas. Modelos de Machine Learning podem capturar relações não lineares, mas correlação e importância de variável não devem ser interpretadas como causalidade clínica.

## 6. Limpeza dos Dados

A limpeza será conservadora. Valores extremos podem ter relevância clínica e não serão removidos automaticamente. Valores ausentes serão tratados dentro do pipeline.

In [ ]:
dados_limpos = dados.copy()
antes = len(dados_limpos)
dados_limpos = dados_limpos.drop_duplicates().reset_index(drop=True)
depois = len(dados_limpos)

print(f"Registros antes: {antes}")
print(f"Registros após remoção de duplicados: {depois}")
print(f"Duplicados removidos: {antes - depois}")

dados_limpos = dados_limpos[dados_limpos[TARGET_COLUMN].isin([NEGATIVE_LABEL, POSITIVE_LABEL])].reset_index(drop=True)

print("Distribuição após limpeza:")
display(dados_limpos[TARGET_COLUMN].value_counts().to_frame("quantidade"))

In [ ]:
checks = []
if "age" in dados_limpos.columns:
    checks.append({
        "variavel": "age",
        "menor_que_zero": int((dados_limpos["age"] < 0).sum()),
        "maior_que_120": int((dados_limpos["age"] > 120).sum()),
    })

for col in ["TSH", "T3", "TT4", "T4U", "FTI", "TBG"]:
    if col in dados_limpos.columns:
        checks.append({
            "variavel": col,
            "menor_que_zero": int((dados_limpos[col] < 0).sum()),
            "maior_que_120": np.nan,
        })

display(pd.DataFrame(checks))

A remoção de duplicados reduz risco de viés. Outliers foram mantidos porque podem representar sinais clínicos importantes, especialmente em exames laboratoriais.

## 7. Separação entre Variáveis Preditoras e Alvo

As variáveis preditoras são as entradas do modelo. A variável-alvo é aquilo que o modelo deve aprender a prever.

In [ ]:
X = dados_limpos.drop(columns=[TARGET_COLUMN])
y = dados_limpos[TARGET_COLUMN].map({NEGATIVE_LABEL: 0, POSITIVE_LABEL: 1})

print(f"Formato de X: {X.shape}")
print(f"Formato de y: {y.shape}")
print("Mapeamento: 0 = negative, 1 = hypothyroid")
display(y.value_counts().rename(index={0: NEGATIVE_LABEL, 1: POSITIVE_LABEL}).to_frame("quantidade"))

## 8. Separação em Treino, Validação e Teste

Usaremos 70% treino, 15% validação e 15% teste, com estratificação para preservar a proporção das classes. O teste será usado apenas na avaliação final.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

splits_summary = pd.DataFrame({
    "conjunto": ["treino", "validação", "teste"],
    "quantidade": [len(X_train), len(X_val), len(X_test)],
    "percentual": [len(X_train)/len(X), len(X_val)/len(X), len(X_test)/len(X)],
    "positivos": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
})
splits_summary["percentual"] = (splits_summary["percentual"] * 100).round(2)
display(splits_summary)

## 9. Pipeline de Pré-processamento

O pipeline evita data leakage porque imputação, padronização e codificação são ajustadas apenas no treino. Variáveis numéricas e categóricas exigem tratamentos diferentes.

In [ ]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Variáveis numéricas:", numeric_features)
print("Variáveis categóricas:", categorical_features)

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(scale_numeric=True):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(numeric_steps)
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])

    return ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])

## 10. Modelagem

Serão comparados três modelos: Regressão Logística, Random Forest e HistGradientBoostingClassifier. O modelo de boosting utilizará `early_stopping` para interromper o treinamento quando novas iterações não melhorarem a validação interna.

In [ ]:
modelos = {
    "Regressão Logística": Pipeline([
        ("preprocessor", build_preprocessor(scale_numeric=True)),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", build_preprocessor(scale_numeric=False)),
        ("classifier", RandomForestClassifier(
            n_estimators=400, min_samples_leaf=2, class_weight="balanced_subsample",
            random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
    "HistGradientBoosting com Early Stopping": Pipeline([
        ("preprocessor", build_preprocessor(scale_numeric=False)),
        ("classifier", HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
            early_stopping=True, validation_fraction=0.15,
            n_iter_no_change=15, random_state=RANDOM_STATE
        )),
    ]),
}

In [ ]:
def get_positive_scores(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X_data)
    return None


def evaluate_model(model, X_data, y_true):
    y_pred = model.predict(X_data)
    y_score = get_positive_scores(model, X_data)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_score) if y_score is not None else np.nan,
    }
    return metrics, y_pred, y_score


def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=[NEGATIVE_LABEL, POSITIVE_LABEL],
        yticklabels=[NEGATIVE_LABEL, POSITIVE_LABEL],
    )
    plt.title(title)
    plt.xlabel("Classe prevista")
    plt.ylabel("Classe real")
    plt.tight_layout()
    plt.show()

In [ ]:
resultados = []
modelos_treinados = {}
validacao_predicoes = {}

for nome, pipeline in modelos.items():
    print(f"Treinando: {nome}")
    pipeline.fit(X_train, y_train)
    modelos_treinados[nome] = pipeline

    train_metrics, _, _ = evaluate_model(pipeline, X_train, y_train)
    val_metrics, y_val_pred, y_val_score = evaluate_model(pipeline, X_val, y_val)
    validacao_predicoes[nome] = {"pred": y_val_pred, "score": y_val_score}

    classifier = pipeline.named_steps["classifier"]
    resultados.append({
        "modelo": nome,
        "accuracy_treino": train_metrics["accuracy"],
        "recall_treino": train_metrics["recall"],
        "f1_treino": train_metrics["f1"],
        "accuracy_validacao": val_metrics["accuracy"],
        "precision_validacao": val_metrics["precision"],
        "recall_validacao": val_metrics["recall"],
        "f1_validacao": val_metrics["f1"],
        "auc_validacao": val_metrics["auc"],
        "iteracoes_usadas": getattr(classifier, "n_iter_", np.nan),
    })

resultados_df = pd.DataFrame(resultados).sort_values(
    by=["recall_validacao", "f1_validacao", "auc_validacao"],
    ascending=False,
).reset_index(drop=True)

display(resultados_df.style.format({col: "{:.3f}" for col in resultados_df.select_dtypes(include=[np.number]).columns}))

## 11. Avaliação dos Modelos

Usaremos accuracy, precision, recall, F1-score, AUC, matriz de confusão e curva ROC. Em saúde, recall é central porque falso negativo pode significar deixar de sinalizar uma pessoa possivelmente doente.

In [ ]:
metrics_long = resultados_df.melt(
    id_vars="modelo",
    value_vars=["accuracy_validacao", "precision_validacao", "recall_validacao", "f1_validacao", "auc_validacao"],
    var_name="metrica",
    value_name="valor",
)

plt.figure(figsize=(13, 6))
sns.barplot(data=metrics_long, x="metrica", y="valor", hue="modelo")
plt.title("Comparação das métricas no conjunto de validação")
plt.xlabel("Métrica")
plt.ylabel("Valor")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.legend(title="Modelo", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
for nome, valores in validacao_predicoes.items():
    print("=" * 80)
    print(nome)
    print(classification_report(y_val, valores["pred"], target_names=[NEGATIVE_LABEL, POSITIVE_LABEL], zero_division=0))
    plot_confusion_matrix(y_val, valores["pred"], f"Matriz de confusão - Validação - {nome}")

In [ ]:
plt.figure(figsize=(9, 7))
for nome, valores in validacao_predicoes.items():
    y_score = valores["score"]
    if y_score is None:
        continue
    fpr, tpr, _ = roc_curve(y_val, y_score)
    model_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f"{nome} (AUC = {model_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
plt.title("Curvas ROC no conjunto de validação")
plt.xlabel("Taxa de falsos positivos")
plt.ylabel("Taxa de verdadeiros positivos / Recall")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

As matrizes de confusão mostram falsos positivos e falsos negativos. A curva ROC avalia separação entre classes em diferentes limiares. Accuracy sozinha é insuficiente em bases desbalanceadas.

## 12. Early Stopping e Análise de Overfitting

O early stopping foi aplicado ao boosting. A ideia é interromper o treinamento quando novas iterações deixam de melhorar o desempenho de validação interna. Curvas de aprendizado ajudam a comparar treino e validação.

In [ ]:
hgb_model = modelos_treinados["HistGradientBoosting com Early Stopping"].named_steps["classifier"]

print(f"Iterações máximas configuradas: {hgb_model.max_iter}")
print(f"Iterações efetivamente usadas: {hgb_model.n_iter_}")
print(f"Early stopping ativado: {hgb_model.early_stopping}")

if hasattr(hgb_model, "validation_score_") and hgb_model.validation_score_ is not None:
    plt.figure(figsize=(10, 5))
    plt.plot(hgb_model.validation_score_, marker="o")
    plt.title("Evolução do score de validação interna - Early Stopping")
    plt.xlabel("Iteração")
    plt.ylabel("Score de validação interna")
    plt.tight_layout()
    plt.show()
else:
    print("O modelo não expôs validation_score_. n_iter_ ainda indica o ponto de parada.")

In [ ]:
for nome, modelo in modelos_treinados.items():
    print(f"Gerando learning curve para: {nome}")
    train_sizes, train_scores, val_scores = learning_curve(
        modelo, X_train, y_train, cv=3, scoring="f1",
        train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1
    )

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Treino")
    plt.plot(train_sizes, val_scores.mean(axis=1), marker="s", label="Validação cruzada")
    plt.fill_between(train_sizes, train_scores.mean(axis=1)-train_scores.std(axis=1), train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=0.15)
    plt.fill_between(train_sizes, val_scores.mean(axis=1)-val_scores.std(axis=1), val_scores.mean(axis=1)+val_scores.std(axis=1), alpha=0.15)
    plt.title(f"Learning Curve - {nome}")
    plt.xlabel("Quantidade de amostras de treino")
    plt.ylabel("F1-score")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
rf_pipeline = modelos["Random Forest"]
param_range = [50, 100, 200, 400]
train_scores, val_scores = validation_curve(
    rf_pipeline, X_train, y_train,
    param_name="classifier__n_estimators",
    param_range=param_range, cv=3, scoring="f1", n_jobs=-1
)

plt.figure(figsize=(8, 5))
plt.plot(param_range, train_scores.mean(axis=1), marker="o", label="Treino")
plt.plot(param_range, val_scores.mean(axis=1), marker="s", label="Validação cruzada")
plt.title("Validation Curve - Random Forest")
plt.xlabel("Número de árvores")
plt.ylabel("F1-score")
plt.legend()
plt.tight_layout()
plt.show()

Se o desempenho de treino for muito superior ao de validação, há sinal de overfitting. Se ambos forem baixos, pode haver underfitting. O early stopping ajuda o boosting a parar quando a validação interna deixa de melhorar.

## 13. Escolha do Melhor Modelo

A seleção usa apenas o conjunto de validação, priorizando recall, F1-score, AUC e estabilidade entre treino e validação.

In [ ]:
resultados_df["gap_f1_treino_validacao"] = (resultados_df["f1_treino"] - resultados_df["f1_validacao"]).abs()

ranking = resultados_df.sort_values(
    by=["recall_validacao", "f1_validacao", "auc_validacao", "gap_f1_treino_validacao"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

display(ranking.style.format({col: "{:.3f}" for col in ranking.select_dtypes(include=[np.number]).columns}))

best_model_name = ranking.loc[0, "modelo"]
best_model = modelos_treinados[best_model_name]
print(f"Melhor modelo selecionado: {best_model_name}")

## 14. Avaliação Final no Conjunto de Teste

O teste simula dados ainda não vistos e só será usado depois da escolha do modelo.

In [ ]:
test_metrics, y_test_pred, y_test_score = evaluate_model(best_model, X_test, y_test)
test_results = pd.DataFrame([test_metrics], index=[best_model_name]).T.rename(columns={best_model_name: "valor"})
display(test_results.style.format("{:.3f}"))

print("Classification report final:")
print(classification_report(y_test, y_test_pred, target_names=[NEGATIVE_LABEL, POSITIVE_LABEL], zero_division=0))

plot_confusion_matrix(y_test, y_test_pred, f"Matriz de confusão final - {best_model_name}")

In [ ]:
if y_test_score is not None:
    fpr, tpr, _ = roc_curve(y_test, y_test_score)
    test_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, linewidth=2, label=f"{best_model_name} (AUC = {test_auc:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
    plt.title("Curva ROC final no conjunto de teste")
    plt.xlabel("Taxa de falsos positivos")
    plt.ylabel("Taxa de verdadeiros positivos / Recall")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    print("Modelo sem score probabilístico disponível para ROC.")

## 15. Interpretação do Modelo

A interpretação será feita com importância por permutação, que avalia a queda no desempenho quando cada variável é embaralhada. Também há interpretação por coeficientes se a Regressão Logística for o melhor modelo e SHAP opcional.

In [ ]:
preprocessor = best_model.named_steps["preprocessor"]
classifier = best_model.named_steps["classifier"]

perm_importance = permutation_importance(
    best_model, X_val, y_val, n_repeats=10,
    random_state=RANDOM_STATE, scoring="f1", n_jobs=-1
)

importance_df = pd.DataFrame({
    "variavel": X_val.columns,
    "importancia_media": perm_importance.importances_mean,
    "importancia_desvio": perm_importance.importances_std,
}).sort_values("importancia_media", ascending=False)

display(importance_df.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(15), x="importancia_media", y="variavel", color="#7E57C2")
plt.title(f"Importância por permutação - {best_model_name}")
plt.xlabel("Queda média no F1-score ao embaralhar a variável")
plt.ylabel("Variável")
plt.tight_layout()
plt.show()

In [ ]:
def get_feature_names(preprocessor):
    feature_names = []
    if numeric_features:
        feature_names.extend(numeric_features)
    if categorical_features:
        cat_pipeline = preprocessor.named_transformers_["cat"]
        encoder = cat_pipeline.named_steps["onehot"]
        feature_names.extend(encoder.get_feature_names_out(categorical_features).tolist())
    return feature_names


if isinstance(classifier, LogisticRegression):
    feature_names = get_feature_names(preprocessor)
    coefficients = pd.DataFrame({
        "atributo": feature_names,
        "coeficiente": classifier.coef_[0],
    })
    coefficients["coeficiente_abs"] = coefficients["coeficiente"].abs()
    coefficients = coefficients.sort_values("coeficiente_abs", ascending=False)
    display(coefficients.head(20))

    plt.figure(figsize=(10, 7))
    sns.barplot(data=coefficients.head(20), x="coeficiente", y="atributo", palette="vlag")
    plt.axvline(0, color="black", linewidth=1)
    plt.title("Coeficientes mais relevantes - Regressão Logística")
    plt.xlabel("Coeficiente")
    plt.ylabel("Atributo")
    plt.tight_layout()
    plt.show()
else:
    print("O melhor modelo não é Regressão Logística; coeficientes lineares não se aplicam.")

In [ ]:
try:
    import shap

    feature_names = get_feature_names(preprocessor)
    X_val_transformed = preprocessor.transform(X_val)
    sample_size = min(250, X_val_transformed.shape[0])
    X_shap = X_val_transformed[:sample_size]

    if isinstance(classifier, (RandomForestClassifier, HistGradientBoostingClassifier)):
        explainer = shap.Explainer(classifier, X_shap, feature_names=feature_names)
        shap_values = explainer(X_shap)
        shap.plots.beeswarm(shap_values, max_display=15)
    else:
        print("SHAP foi carregado, mas este bloco está configurado principalmente para modelos de árvore/boosting.")
except Exception as exc:
    print("SHAP não foi executado neste ambiente.")
    print(f"Motivo: {type(exc).__name__}: {exc}")

As variáveis mais importantes ajudam a entender o comportamento do modelo, mas não provam causalidade. Em saúde, qualquer interpretação deve ser validada por conhecimento clínico.

## 16. Salvamento e Carregamento do Modelo

O melhor pipeline será salvo completo, incluindo pré-processamento e classificador. Isso facilita uso posterior sem repetir manualmente as transformações.

In [ ]:
model_path = MODELS_DIR / "modelo_classificacao_tireoide.pkl"
joblib.dump(best_model, model_path)
print(f"Modelo salvo em: {model_path.resolve()}")

In [ ]:
modelo_carregado = joblib.load(model_path)
print("Modelo carregado com sucesso.")
print(modelo_carregado)

## 17. Relatório Final e Discussão dos Resultados

Este projeto desenvolveu uma solução inicial de Machine Learning para apoiar a análise de dados médicos tabulares relacionados à tireoide. O objetivo foi classificar registros entre casos de hipotireoidismo e resultados negativos, utilizando variáveis clínicas, demográficas e laboratoriais.

A análise exploratória indicou que o dataset possui variáveis numéricas relevantes, como idade e exames laboratoriais, além de variáveis categóricas associadas a histórico clínico e indicação de exames medidos. Também foi observado desbalanceamento significativo entre as classes, com maior quantidade de registros negativos do que positivos para hipotireoidismo. Esse ponto influenciou diretamente a escolha das métricas, pois a acurácia isolada poderia mascarar dificuldades do modelo em identificar a classe minoritária.

Na etapa de limpeza, registros duplicados foram removidos para reduzir risco de viés na avaliação. Valores ausentes foram tratados dentro dos pipelines, evitando data leakage. Valores extremos não foram removidos automaticamente, pois em dados médicos eles podem representar situações clinicamente importantes.

Foram testados três modelos: Regressão Logística, Random Forest e HistGradientBoostingClassifier com early stopping. A comparação foi feita no conjunto de validação, considerando principalmente recall, F1-score, AUC e estabilidade entre treino e validação. O early stopping foi incluído no modelo de boosting para interromper o aprendizado quando novas iterações não trouxessem melhoria relevante, ajudando a reduzir risco de overfitting.

A avaliação final no conjunto de teste foi realizada somente após a escolha do melhor modelo. Esse conjunto simulou dados ainda não vistos e permitiu estimar a capacidade de generalização da solução. As matrizes de confusão, curvas ROC, learning curves e gráficos de métricas auxiliaram a entender não apenas o desempenho médio, mas também os erros mais relevantes em contexto clínico.

A interpretação do modelo foi conduzida com técnicas como importância por permutação, coeficientes quando aplicável e SHAP de forma opcional. Essas análises ajudam a identificar quais variáveis influenciaram mais as previsões, mas não devem ser confundidas com relações causais. A interpretação clínica exige validação por especialistas e análise contextual.

Como limitações, este notebook utiliza uma base tabular específica e não contempla validação prospectiva, integração com prontuários reais, análise de vieses populacionais ou avaliação regulatória. Em ambiente real, falsos negativos podem atrasar investigação clínica, enquanto falsos positivos podem gerar preocupação e exames adicionais. Portanto, o modelo deve ser usado apenas como apoio à triagem ou análise inicial.

Como melhorias futuras, seria possível testar calibração de probabilidades, ajuste de limiar de decisão, validação externa, técnicas específicas para dados desbalanceados, monitoramento de drift e integração com protocolos clínicos. A conclusão principal é que Machine Learning pode apoiar a organização e priorização da análise médica, mas não deve substituir a avaliação de profissionais da saúde, que sempre devem ter a decisão final.